## $\chi^2$ optimization for the the chunk waterfall plot

Notebook for running the $\chi^2$ on the chunck waterfall maps.

   $\bullet$ Import list - holds all the imports including the parameter file with all the location information

In [1]:
%load_ext autoreload
%autoreload 2

%reload_ext autoreload

from import_list import *

In [6]:
# File name
print ('File name: '+param.file_name)

# Time of observation
tools.timepoint(fname=int(param.file_name), date=None)
 
# Katdal information
katdal_info = pickle.load(open(param.telescope_info_path, 'rb'), encoding='latin1')
info = [katdal_info[i] for i in katdal_info.keys()]
nd_s0=katdal_info['nd_s0']
nd_s0_coords=katdal_info['nd_s0_coords']
frequency=katdal_info['frequency']

# Frequency start and end point
print ('\nFrequency range from {fs}-{fe} MHz'.format(fs=param.fs, fe=param.fe))

# Bias count for number of constellations
bias = np.ones(len(param.constellations))

# Folder for saving the current work (eg: yyyy_mm_dd)
print ('Folder for saving data is: '+param.work_folder)

# Frequency band inside the frequency range  [MHz]
fband_s, fband_e = 1100, 1350

File name: 1551055211
Date of observation: 2019-02-25 02:40:11
Fname: 1551055211

Frequency range from 1000-1500 MHz
Folder for saving data is: parallel_2022_03_20


In [9]:
def radiometer_eq(data):
    '''
    Radiometer euquation for determining the error on the data
    '''
    d_nu = 0.2 * 10**6 # Hz
    d_t = 2 # s
    n_pol = 2 
    n_dish = 58
    sig2 = data**2 / (n_pol*d_nu*d_t*n_dish)
    sig = np.sqrt(sig2)
    
    return sig


In [15]:
# Initalizing the class 
sat = ss(file_name=param.file_name, 
         sats_only=None, 
         data_loc=param.meerkat_data, 
         sat_loc=param.meerkat_data,
         survey_info=[nd_s0, nd_s0_coords, frequency], 
         sat_info=param.satellite_catalogue,
         plots_loc=param.save_plots,
         sat_beam=param.beam_model, 
         frequency_range=[param.fs,param.fe], 
         constellations=param.constellations, 
         nearby_satellites=param.nearby_constellation_fpath)

# Creating random alpha values for the inital excecution
np.random.seed()
dic = 10*np.random.random(sat.alpha_len)

ts=0

# Running the excecution
sat.excecute(a_param=dic,
             obs_time_start=nd_s0[ts*100], obs_time_end=nd_s0[ts*100+100], 
             obs_frequency_start=fband_s, obs_frequency_end=fband_e, 
             file_bias_choice=bias, 
             add_sub=[1, 1],
             band_lvl=[None, None], 
             bandsize=None)

In [21]:
def sat_sim_p(ts, queue):
    '''
    sat_sim - function used for the parellization optimization of the data.
    ts - the chunk number, which relates to the timestamp
    queue - multiprocessing function
    '''
    # Creating different chunks that refer to different timestamps
    num = 'chunk'+'_'+str(ts)
    print (num)
    
    # 1st initialization
    sat = ss(file_name=param.file_name, 
             sats_only=None, 
             data_loc=param.meerkat_data, 
             sat_loc=param.meerkat_data,
             survey_info=[nd_s0, nd_s0_coords, frequency], 
             sat_info=param.satellite_catalogue,
             plots_loc=param.save_plots,
             sat_beam=param.beam_model, 
             frequency_range=[param.fs,param.fe], 
             constellations=param.constellations, 
             nearby_satellites=param.nearby_constellation_fpath)
    

    np.random.seed()
    dic = 10*np.random.random(sat.alpha_len)

    sat.excecute(a_param=dic,
                 obs_time_start=nd_s0[ts*100], obs_time_end=nd_s0[ts*100+100], 
                 obs_frequency_start=fband_s, obs_frequency_end=fband_e, 
                 file_bias_choice=bias, 
                 add_sub=[1, 1],
                 band_lvl=[None, None], 
                 bandsize=None)

     # Note!! Add a conditon here for flagging, if enough of the data is flagged then ignore the data    

    # Chi^2 fucntion for the optimization
    def chisq_func2(a_param):
        """
        Chi2 function which will take in all the parameters for the satellites
        """

        sat.excecute(a_param=dic,
                     obs_time_start=nd_s0[ts*100], obs_time_end=nd_s0[ts*100+100], 
                     obs_frequency_start=fband_s, obs_frequency_end=fband_e, 
                     file_bias_choice=bias, 
                     add_sub=[1, 1],
                     band_lvl=[None, None], 
                     bandsize=None)


        # Adding the mask to the chi^2 optimization
        
        # ----------No mask
        # simulation = sat.simulation_TOD_slice
        # data = sat.calibration_data_slice
        
        # ----------Masked
        simulation = np.ma.array(data=sat.simulation_TOD_slice, mask=sat.mask_nearby_satellites_slice.T)
        data = np.ma.array(data=sat.calibration_data_slice, mask=sat.mask_nearby_satellites_slice.T)
        
        # Radiometer equation
        sig = radiometer_eq(data=data)    

        # Chi^2 equation
        chi_sq = np.ma.sum( (simulation - data )**2 / sig**2)
        return chi_sq

    # Setting the bound values for the optimization
    bnd_val = (0.0, 30)
    bnds = [bnd_val for bnd_i in range(sat.alpha_len)]

    # Running the optimization
    print ('Running optimization: '+num)
    signal_PL = opt.minimize(fun=chisq_func2,          # The chi^2 equation for optimization
                             x0=dic,                   # The param values for the optimzation
                             method='Powell',          # The method of choice in the optimization
                             bounds=bnds,              # Bound values for the optimization
                             tol=1e-6,                 # Tolerance value
                             options={'maxiter':20})

    print ('Running 2nd init')
    # Initializing the class a second time tp excecute with the best fit values
    sat2 = ss(file_name=param.file_name, 
              sats_only=None, 
              data_loc=param.meerkat_data, 
              sat_loc=param.meerkat_data,
              survey_info=[nd_s0, nd_s0_coords, frequency], 
              sat_info=param.satellite_catalogue,
              plots_loc=param.save_plots,
              sat_beam=param.beam_model, 
              frequency_range=[param.fs,param.fe], 
              constellations=param.constellations, 
              nearby_satellites=param.nearby_constellation_fpath)


    sat2.excecute(a_param=signal_PL.x,                                          # Applying the best-fit paramaters
                  obs_time_start=nd_s0[ys*100], obs_time_end=nd_s0[ts*100+100], 
                  obs_frequency_start=fband_s, obs_frequency_end=fband_e, 
                  file_bias_choice=bias,
                  add_sub=[1, 1], 
                  band_lvl=[None, None], 
                  bandsize=None)
    
    # Storing the data
    print ('Storing data: '+num)
    data_info = {'initial':dic,                                                 # Intial parameter test
                 'time': sat.nd_s0[ts*100:ts*100+100+1],                        # The time range for the observation
                 'best-fit':signal_PL.x,                                        # Best_fit from the optimization 
                 'chi2_value':signal_PL.fun,                                    # The chi^2 value
                 'chi2_div':signal_PL.fun/sat2.simulation_TOD_slice.size        # chi^2 value divided by number of data points
    }

    pickle.dump(data_info, open(param.chi2_save_data+'/data_test_full_mask_'+str(num)+'.p', 'wb'))


    print (num+' End')
    

In [4]:
print ('Number of cpu\'s available: '+str(mp.cpu_count()))

Number of cpu's available: 32


In [23]:
# Running the parallelization for the data

if __name__ == '__main__':

    procs =1
    
    queue = mp.Queue()
    
    processes = [mp.Process(target=sat_sim_p,  args=(intv, queue)) for intv in range(0,procs,1)]
    
    
    for prs in processes:
        prs.start()

    for prs in processes:
        prs.join()

chunk_0
